# Empacotamento e publicação no Hugging Face Hub (estágio 16)

Publica a versão de divulgação da fonte escolhida: o conjunto de patches (imagens, máscaras, manifesto com caminhos relativos e estatísticas de normalização) em um repositório de **dataset** (CC BY 4.0) e os pesos por dobra da **U-Net** e do **SegFormer** em repositórios de **modelo** (MIT), com model cards e metadados de treino/avaliação. A autenticação ocorre nesta célula com `HF_TOKEN`; o token nunca é impresso nem gravado no repositório. Por padrão os repositórios são públicos e derivados da conta autenticada, podendo ser sobrescritos por `HF_DATASET_REPO`, `HF_UNET_REPO` e `HF_SEGFORMER_REPO`; `HF_RELEASE_PRIVATE=true` cria repositórios privados e `HF_RELEASE_FORCE=true` força o reenvio.

## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.

In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/jotap1101/tcc-lab/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")

## Dependências pinadas

Garante as versões fixadas em `requirements-runtime.txt`, o mesmo conjunto nas duas plataformas. Quando uma biblioteca com extensão compilada já importada (numpy, scipy, torch, entre outras) diverge da versão pinada, o kernel é reiniciado automaticamente para evitar inconsistência de ABI; nesse caso, o notebook é reexecutado a partir da primeira célula.

In [ ]:
# Garante as versões pinadas e reinicia o kernel quando a ABI carregada divergir.
from src.setup import install_runtime_requirements

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
install_runtime_requirements(requirements)

## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve os caminhos canônicos do dataset, dos modelos, das estatísticas e dos metadados de release.

In [ ]:
# Resolve o contexto de runtime e os caminhos canônicos do release.
from src.data.source_decision import chosen_source
from src.release import release_dir, release_metadata_path
from src.setup import initialize

ctx = initialize(workspace)
platform = ctx.platform
storage_paths = ctx.storage_paths
config = ctx.config
source_name = chosen_source()
print(f"Plataforma: {platform}")
print(f"Fonte de treinamento/avaliação (decisão manual): {source_name}")
print(f"Diretório do release: {release_dir(storage_paths, source_name)}")
print(f"Metadata do release: {release_metadata_path(storage_paths, source_name)}")

## Reprodutibilidade

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch, além de registrar as versões do ambiente. O empacotamento apenas copia patches e pesos existentes e envia ao Hugging Face Hub; não há treinamento nem inferência de modelos.

In [ ]:
# Fixa sementes, flags determinísticas e registra o ambiente.
from src.utils import log_environment, setup_reproducibility

setup_reproducibility(config)
log_environment()

## Dependências dos estágios 06, 08, 09, 10 e 11

Verifica o manifesto, as estatísticas de normalização, os pesos e artefatos de treino dos dois modelos e os resumos vigentes da avaliação. O release não retreina nem reavalia: usa exatamente os artefatos validados pelos fingerprints.

In [ ]:
# Valida manifesto, estatísticas, treinos e avaliações antes da publicação.
from src.data.eda import normalization_stats_path
from src.data.patch_generation import manifest_path
from src.evaluation import evaluation_summary_path
from src.release import validate_inputs
from src.utils import check_dependencies

check_dependencies(
    {
        "Manifesto (estágio 06)": manifest_path(storage_paths, source_name),
        "Estatísticas de normalização (estágio 08)": normalization_stats_path(
            storage_paths, source_name
        ),
        "Resumo da U-Net (estágio 11)": evaluation_summary_path(
            storage_paths, "unet", source_name
        ),
        "Resumo do SegFormer (estágio 11)": evaluation_summary_path(
            storage_paths, "segformer", source_name
        ),
    }
)
summaries = validate_inputs(storage_paths, source_name)
print(f"Patches avaliados: {summaries['unet']['n_patches']}")
print(f"Dobras: {len(summaries['unet']['per_fold'])}")

## Autenticação no Hugging Face

Lê `HF_TOKEN` de variável de ambiente ou dos Secrets da plataforma e autentica a conta. O token é exigido apenas para escrita nesta sessão e não é impresso; a conta retornada define os nomes padrão dos repositórios.

In [ ]:
# Autentica no Hugging Face Hub e obtém o nome da conta (token nunca é impresso).
from src.release import authenticate_huggingface

hf_token, hf_username = authenticate_huggingface()
print(f"Conta Hugging Face: {hf_username}")

## Repositórios de destino

Define os repositórios de dataset e de cada modelo. Por padrão, derivam da conta autenticada e do nome da fonte; podem ser sobrescritos pelas variáveis `HF_DATASET_REPO`, `HF_UNET_REPO` e `HF_SEGFORMER_REPO` para publicar em outra organização ou namespace.

In [ ]:
# Resolve os IDs dos repositórios (com sobrescrita opcional por variáveis de ambiente).
import os

from src.release import resolve_repo_ids

repos = resolve_repo_ids(
    hf_username,
    source_name,
    dataset_repo=os.getenv("HF_DATASET_REPO"),
    unet_repo=os.getenv("HF_UNET_REPO"),
    segformer_repo=os.getenv("HF_SEGFORMER_REPO"),
)
for key, repo_id in repos.items():
    print(f"{key}: {repo_id}")

## Empacotamento e publicação

Cria os três repositórios (dataset e um por modelo), copia os patches e pesos para um staging temporário, grava o manifesto com caminhos relativos, as estatísticas, os model cards e o `model_config.json`, e envia ao Hugging Face Hub. A publicação é reutilizada quando o fingerprint de dataset, pesos, configuração e repositórios coincide e os arquivos remotos estão presentes. O staging é temporário e requer espaço local para os patches; links rígidos são usados quando possível para evitar cópias.

In [ ]:
# Empacota e publica (ou reutiliza) o dataset e os modelos no Hugging Face Hub.
import os

from src.release import run_release

private = os.getenv("HF_RELEASE_PRIVATE", "false").strip().lower() in {"1", "true", "yes"}
force = os.getenv("HF_RELEASE_FORCE", "false").strip().lower() in {"1", "true", "yes"}
release_result = run_release(
    storage_paths,
    source_name=source_name,
    dataset_repo=repos["dataset"],
    unet_repo=repos["unet"],
    segformer_repo=repos["segformer"],
    token=hf_token,
    private=private,
    force=force,
)
print(f"Release reutilizado: {release_result.skipped}")
print(f"Patches publicados: {release_result.n_patches}")

## Resumo da publicação

Exibe os repositórios publicados, os commits e os caminhos do metadata de auditoria. Os links podem ser usados diretamente na monografia e nos materiais de divulgação.

In [ ]:
# Exibe o resumo do release e os links dos repositórios/commits.
from src.release import repo_url
from src.utils import print_summary

summary = {
    "Fonte": source_name,
    "Dataset": repo_url(release_result.repos["dataset"], "dataset"),
    "Modelo U-Net": repo_url(release_result.repos["unet"], "model"),
    "Modelo SegFormer": repo_url(release_result.repos["segformer"], "model"),
    "Patches publicados": release_result.n_patches,
    "Commit do dataset": release_result.commit_urls.get("dataset"),
    "Commit da U-Net": release_result.commit_urls.get("unet"),
    "Commit do SegFormer": release_result.commit_urls.get("segformer"),
    "Metadata": str(release_result.release_path),
}
print_summary(summary, "16")